# 📊 Master Geometric Data Augmentation & Imbalance Classification Benchmark
## Comprehensive Comparative Analysis of Experiments 1 to 6

### 📌 Study Background & Rationale
Severe Acute Malnutrition (**SAM**) represents only **3.5% (220 specimens)** of our dataset ($N=6,335$). Standard classification accuracy is misleading in imbalanced contexts because a naive model predicting 100% ONC achieves 85.1% accuracy while completely failing to detect SAM cases.

To scientifically determine the optimal balancing strategy, we evaluated **6 distinct data augmentation & balancing pipelines** across **Stratified 5-Fold Cross-Validation** using **XGBoost**:
1. **Experiment 1 (Baseline)**: Full Data ($5,393$-$722$-$220$) + Class-Weighted XGBoost.
2. **Experiment 2 (Real Data Benchmark)**: Pure Real Subset ($220$-$220$-$220$), 100% biological data.
3. **Experiment 3 (Limited Synthetic)**: $500$-$500$-$500$ ($18.7\%$ synthetic SAM).
4. **Experiment 4 (Preferred Balanced)**: $722$-$722$-$722$ ($23.2\%$ synthetic SAM, preserving 100% real MAM & SAM).
5. **Experiment 5 (SMOTE-Tomek)**: Full Data + SMOTE-Tomek (Training fold only).
6. **Experiment 6 (Borderline-SMOTE)**: Full Data + Borderline-SMOTE (Training fold only).

*Note: In all experiments, noisy & collinear landmarks (**LM11, LM21, LM22**) were removed to optimize geometric shape signal.*


In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = r"e:\ap-govt-EDA-model-tainning-testing\GM_out_of_sample_classification-main\GM_out_of_sample_classification-ap-govt\geometric-dataAugmentation"
res_file = os.path.join(data_dir, "all_experiments_results.csv")
cm_file = os.path.join(data_dir, "confusion_matrices.json")

# Load Results
df_res = pd.read_csv(res_file)
with open(cm_file, 'r') as f:
    cms_dict = json.load(f)

print("🏆 MASTER EXPERIMENTAL EVALUATION TABLE")
print("="*85)
display(df_res)


🏆 MASTER EXPERIMENTAL EVALUATION TABLE
             Experiment                Data Used          Synthetic %  SAM Recall  SAM Precision  SAM F1  Macro F1  Balanced Acc
        Exp 1: Baseline             5393-722-220                 0.0%      0.2409         0.6611  0.3527    0.6081        0.5584
 Exp 2: Real Only (220)              220-220-220                 0.0%      0.6773         0.6328  0.6530    0.6415        0.6424
    Exp 3: Balanced 500              500-500-500                18.7%      0.5682         0.5494  0.5581    0.6586        0.6601
   Exp 4: Preferred 722              722-722-722                23.2%      0.4909         0.6263  0.5473    0.6837        0.6733
     Exp 5: SMOTE-Tomek Full Data (5393-722-220) Dynamic Oversampling      0.4682         0.5253  0.4948    0.6705        0.6618
Exp 6: Borderline-SMOTE Full Data (5393-722-220) Dynamic Oversampling      0.4045         0.5869  0.4771    0.6621        0.6397

In [ ]:
# Plot Comparative Performance Metrics
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: SAM Recall & SAM F1
df_res.plot(x='Experiment', y=['SAM Recall', 'SAM F1', 'SAM Precision'], kind='bar', ax=axes[0], colormap='viridis', width=0.8)
axes[0].set_title("SAM Class Metrics across Experiments", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Score", fontsize=11)
axes[0].set_ylim(0, 0.8)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].set_xticklabels(df_res['Experiment'], rotation=30, ha='right')

# Plot 2: Macro F1 & Balanced Accuracy
df_res.plot(x='Experiment', y=['Macro F1', 'Balanced Acc'], kind='bar', ax=axes[1], colormap='magma', width=0.7)
axes[1].set_title("Overall Model Metrics across Experiments", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Score", fontsize=11)
axes[1].set_ylim(0.4, 0.8)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].set_xticklabels(df_res['Experiment'], rotation=30, ha='right')

plt.tight_layout()
plt.show()


In [ ]:
# Plot Side-by-Side Confusion Matrices (2x3 Grid)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
class_names = ['ONC', 'MAM', 'SAM']

for idx, (exp_name, cm_list) in enumerate(cms_dict.items()):
    cm = np.array(cm_list)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], xticklabels=class_names, yticklabels=class_names, cbar=False)
    axes[idx].set_title(f"{exp_name}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Predicted Class")
    axes[idx].set_ylabel("True Class")

plt.suptitle("Side-by-Side Aggregated Confusion Matrices (5-Fold CV)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 💡 Scientific Conclusions & Strategy Recommendations

### 🥇 1. Highest SAM Sensitivity & Cleanest Benchmark: **Experiment 2 (220-220-220 Real Only)**
* **SAM Recall**: **67.73%** | **SAM F1**: **0.6530** | **SAM Precision**: **63.28%**
* **Why it wins scientifically**: By using equal 220 real specimens for each class, the model learns unbiased shape differences without synthetic artifacts. It achieves the highest SAM Recall (detecting over two-thirds of severe malnutrition cases) and highest SAM F1-Score.

### 🥈 2. Best Machine Learning Tradeoff: **Experiment 4 (722-722-722 Preferred Balanced)**
* **Macro F1**: **0.6837** | **Balanced Acc**: **0.6733** | **SAM Precision**: **62.63%**
* **Why it is preferred for publication**: Retains **100% of available real MAM and SAM specimens**, uses only a moderate amount of synthetic SAM data ($23.2\%$), and delivers high overall macro accuracy while maintaining strong SAM detection.

### 🥉 3. Full Dataset Winner: **Experiment 5 (Full Data + SMOTE-Tomek)**
* **Macro F1**: **0.6705** | **Balanced Acc**: **0.6618** | **SAM Recall**: **46.82%**
* **Why it matters**: Demonstrates that SMOTE-Tomek dynamic oversampling + Tomek link noise removal outperforms standard class-weighted XGBoost (Baseline SAM Recall was only 24.09%).
